题目：根据条件替换 Tensor 中的元素
给定一个二维 tensor `x`，形状为 `(N, M)`，请完成函数 `replace_tensor(x)`：
要求：

1. 如果某个元素 **大于该行平均值**，保留原值
2. 如果某个元素 **小于等于该行平均值**，替换为 **该列的最大值**
3. **只能使用 PyTorch tensor 操作，不允许使用 Python for 循环**

In [4]:
import torch

def replace_tensor(x: torch.Tensor):
    # 行均值：(N, 1) 方便和 (N, M) 广播比较
    row_mean = x.mean(dim=1, keepdim=True)          # (N, 1)

    # 列最大值：(1, M) 方便广播到 (N, M)
    col_max = x.max(dim=0, keepdim=True).values     # (1, M)

    # mask: True 表示保留原值（> 行均值）
    mask = x > row_mean                             # (N, M)

    # 满足条件保留 x，否则用 col_max
    result = torch.where(mask, x, col_max)          # (N, M)
    return result

In [6]:

def replace_tensor(x: torch.Tensor):
    row_mean = x.mean(dim=1, keepdim=True)                 # (N, 1)
    max_x = x.max(dim=0).values.unsqueeze(0).repeat(x.size(0), 1)  # (N, M)
    result = torch.where(x > row_mean, x, max_x)
    return result

In [7]:
%%time
replace_tensor(torch.tensor([[1, 2, 3], [4, 5, 6]],dtype=torch.float32))

CPU times: total: 0 ns
Wall time: 4.74 ms


tensor([[4., 5., 3.],
        [4., 5., 6.]])

给定二维张量 `x`，shape 为 `(N, M)`，实现函数 `normalize_and_fill(x)`，要求：

1. 计算每一行的 **最大值** `row_max`（shape `(N, 1)`）
2. 计算每一列的 **最小值** `col_min`（shape `(1, M)`）
3. 生成输出 `y`（shape `(N, M)`），规则如下（只能用 tensor 操作，不准 for）：
- 若 `x[i, j] == row_max[i]`（即该元素是本行最大值，允许有多个最大值并列），则
	- `y[i, j] = 0`
- 否则：
	- 先把该元素做“按行归一化”：
		$
		 norm = x[i,j] / row\_max[i]
		$
	- 如果 `norm < 0.5`，则用该列最小值替换：
		- `y[i, j] = col_min[j]`
	- 否则保留归一化值：
		- `y[i, j] = norm`

In [2]:
import torch

def normalize_and_fill(x: torch.Tensor):
    """
    x: (N, M), float tensor, 假设 row_max 不为 0
    return: y (N, M)
    """
    # TODO
    row_max=x.max(dim=1,keepdim=True).values
    col_min=x.min(dim=0,keepdim=True).values
    y=torch.where(x/row_max<0.5,col_min,x/row_max)
    y=torch.where(x==row_max,0,y)

    return y

In [17]:
x = torch.tensor([
    [2., 4., 1.],
    [3., 6., 6.]
])
normalize_and_fill(x)

tensor([[0.5000, 0.0000, 1.0000],
        [0.5000, 0.0000, 0.0000]])

给定 `x`（float tensor，shape `(N, M)`）和整数 `k`（`1 <= k <= M`），
要求生成 `y`，规则：

1. 对每一行，找出该行 **最大的 k 个元素的位置**（若有相等值，按 `torch.topk` 默认规则即可）
2. 这些 top-k 位置：`y[i,j] = x[i,j]`（保留原值）
3. 其他位置：`y[i,j] = mean_col[j]`，其中 `mean_col` 是 `x` 的 **按列均值**（shape `(1, M)`）
4. **只能用 PyTorch tensor 操作，不允许 Python for 循环**

提示（不强制）：你可能会用到 `torch.topk`、`scatter_` / `scatter`、`torch.where`、broadcast。

In [ ]:
def keep_topk_fill_mean(x: torch.Tensor, k: int) -> torch.Tensor:  
    """  
    返回 y (N, M)  
    """
    indice=torch.topk(x,k,dim=-1)[1]
    mean_col=x.mean(dim=0,keepdim=True)
    y=torch.zeros_like(x)+mean_col
    
    y.scatter_(dim=-1, index=indice, src=x.gather(dim=-1, index=indice))
    
    return y
    
def keep_topk_fill_mean(x: torch.Tensor, k: int) -> torch.Tensor:  
    """  
    返回 y (N, M)  
    """
    idx = torch.topk(x, k, dim=1).indices
    y = x.mean(dim=0, keepdim=True).expand_as(x).clone()
    y.scatter_(1, idx, x.gather(1, idx))   
    
    return y    


In [29]:
%%time
x = torch.tensor([
    [1., 9., 2., 8.],
    [7., 3., 6., 4.]
])
k = 2
keep_topk_fill_mean(x, k)

CPU times: total: 0 ns
Wall time: 0 ns


tensor([[4., 9., 4., 8.],
        [7., 6., 6., 6.]])